In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-32B-AWQ",
    tensor_parallel_size=1,
    quantization="awq",
    max_model_len=8192,
    trust_remote_code=True,
)

print("✓ Model loaded successfully!")

WARNING 07-01 18:18:00 [config.py:71] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
INFO 07-01 18:18:02 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen3-32B-AWQ'}
INFO 07-01 18:18:03 [model.py:611] Resolved architecture: Qwen3ForCausalLM
INFO 07-01 18:18:03 [model.py:1745] Using max model len 8192
INFO 07-01 18:18:03 [awq_marlin.py:273] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 07-01 18:18:03 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-01 18:18:04 [vllm.py:999] Asynchronous scheduling is enabled.
INFO 07-01 18:18:04 [kernel.py:270] Final 

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=3265) INFO 07-01 18:18:19 [default_loader.py:397] Loading weights took 6.92 seconds
(EngineCore pid=3265) INFO 07-01 18:18:20 [gpu_model_runner.py:5187] Model loading took 18.24 GiB memory and 9.067837 seconds
(EngineCore pid=3265) INFO 07-01 18:18:28 [backends.py:1089] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/e2f8214a4b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3265) INFO 07-01 18:18:28 [backends.py:1148] Dynamo bytecode transform time: 7.51 s
(EngineCore pid=3265) INFO 07-01 18:18:32 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 3.663 s
(EngineCore pid=3265) INFO 07-01 18:18:32 [decorators.py:311] Directly load AOT compilation from path /home/jovyan/.cache/vllm/torch_compile_cache/torch_aot_compile/d63130785d34e0c32e4910307d191df28e2ee6230c522344427983b01aaaed77/rank_0_0/model
(EngineCore pid=3265) INFO 07-01 18:18:32 [monitor.py:53] torch.compile took 12.12 s in total

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:22<00:00,  2.23it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:13<00:00,  2.50it/s]


(EngineCore pid=3265) INFO 07-01 18:19:28 [gpu_model_runner.py:6585] Graph capturing finished in 38 secs, took 1.54 GiB
(EngineCore pid=3265) INFO 07-01 18:19:28 [gpu_worker.py:639] CUDA graph pool memory: 1.54 GiB (actual), 1.49 GiB (estimated), difference: 0.05 GiB (3.0%).
(EngineCore pid=3265) INFO 07-01 18:19:28 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=3265) INFO 07-01 18:19:29 [core.py:306] init engine (profile, create kv cache, warmup model) took 69.39 s (compilation: 12.12 s)
(EngineCore pid=3265) INFO 07-01 18:19:29 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
✓ Model loaded successfully!
(EngineCore pid=3265) WARNING 07-01 18:23:21 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cov

## Prepare a random sample from the training dataframe

In [2]:
import pandas as pd
import json
import re
from vllm import SamplingParams

full_df = pd.read_csv('../data/full_df_with_partitions.csv', index_col=0)
train_df = full_df[full_df['split']=='train']

# Add the labels columns
train_df['label_is_correct'] = train_df['Misconception_Code'].str.contains('Correct', case=False, na=False).copy()
train_df['label_is_misunderstanding'] = ~(
    train_df['label_is_correct'] | 
    train_df['Misconception_Code'].str.contains('Neither', case=False, na=False).copy()
)
# Get a sample
sample_df = train_df.sample(n=100, random_state=42).reset_index(drop=True)

/tmp/ipykernel_153/1194613047.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['label_is_correct'] = train_df['Misconception_Code'].str.contains('Correct', case=False, na=False).copy()
/tmp/ipykernel_153/1194613047.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['label_is_misunderstanding'] = ~(


## Prepare input frame and output schema

In [3]:
import pandas as pd
import json
import re
from pydantic import BaseModel
from vllm import SamplingParams

# Define the exact output schema
class MisconceptionLabel(BaseModel):
    is_correct: bool
    is_misunderstanding: bool
    which_misunderstanding: str

json_schema = MisconceptionLabel.model_json_schema()

MISCONCEPTION_DEFINITION = """
A "misunderstanding" is a systematic error pattern, NOT a simple calculation slip, guess, or vague/unclear response. There are two types. 
Conceptual misunderstandings stem from a wrong mental model of a math concept. Examples: whole-number bias (thinking 3/9 > 2/5 because 3>2 and 9>5; thinking 0.15 > 0.4 because 15>4; thinking zeros after the decimal point don't matter; thinking multiplication always produces a larger number), thinking "=" means "compute an answer" rather than representing equivalence, thinking a variable just means "a missing number" rather than a quantity that can be solved for.
Procedural misunderstandings come from applying the wrong procedure consistently, often by misapplying a rule from one operation to another. Examples: adding fraction numerators and denominators independently like whole numbers (3/5 + 3/5 = 6/10), using a common denominator when multiplying fractions instead of multiplying the denominator (3/5 * 2/5 = 6/5), or inverting the wrong fraction during division.

A misunderstanding must reflect a CONSISTENT, SYSTEMATIC pattern of incorrect reasoning. 
A simple arithmetic slip (e.g., 2 x 8 = 18), a guess, or a vague/unexplained answer is NOT a misunderstanding.
"""

def build_prompt(row):
    return f"""You are analyzing a student's response to a math question to determine 
whether their answer is correct, and if incorrect, whether it reflects a genuine 
mathematical misunderstanding (as opposed to a simple slip, guess, or vague response).

DEFINITION OF MISUNDERSTANDING:
{MISCONCEPTION_DEFINITION}

QUESTION: {row['Question.Text']}
STUDENT'S MULTIPLE CHOICE ANSWER: {row['MC_Answer']}
STUDENT'S SELF-EXPLANATION: {row['StudentExplanation']}

Based on the above, determine:
1. is_correct: Is the student's answer mathematically correct?
2. is_misunderstanding: If incorrect, does the explanation reveal a genuine systematic 
   misunderstanding (per the definition above), as opposed to a calculation slip, guess, 
   or vague/unexplained response?
3. which_misunderstanding: If is_misunderstanding is true, briefly describe the student's thought process that led to the misunderstanding. 
   If is_misunderstanding is false, leave this as an empty string.
"""

# Make a list of conversations
conversations = [
    [{"role": "user", "content": build_prompt(row)}]
    for _, row in sample_df.iterrows()
]

# Define structured output
from vllm.sampling_params import StructuredOutputsParams
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    structured_outputs=StructuredOutputsParams(json=json_schema),
)

# Run model and get results

In [4]:
# Run inference, disabling Qwen3's "thinking" mode so output is just the JSON
outputs = llm.chat(
    conversations,
    sampling_params,
    chat_template_kwargs={"enable_thinking": False},
)

# Parsing should now basically always succeed, but keep a safety net
def parse_response(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(text)
        return {
            "is_correct": parsed.get("is_correct"),
            "is_misunderstanding": parsed.get("is_misunderstanding"),
            "which_misunderstanding": parsed.get("which_misunderstanding", ""),
        }
    except json.JSONDecodeError:
        return {"is_correct": None, "is_misunderstanding": None, "which_misunderstanding": None}

parsed_results = [parse_response(o.outputs[0].text) for o in outputs]

results_df = sample_df.copy()
results_df["is_correct"] = [r["is_correct"] for r in parsed_results]
results_df["is_misunderstanding"] = [r["is_misunderstanding"] for r in parsed_results]
results_df["which_misunderstanding"] = [r["which_misunderstanding"] for r in parsed_results]

# Enforce business rules
results_df.loc[results_df["is_correct"] == True, "is_misunderstanding"] = False
results_df.loc[results_df["is_misunderstanding"] == False, "which_misunderstanding"] = ""

# Sanity check: should now be 0 or near-0
print(f"Remaining parse failures: {results_df['is_correct'].isna().sum()}")
results_df.head(10)

Rendering conversations:   0%|          | 0/100 [00:00<?, ?it/s]

INFO 07-01 18:23:20 [hf.py:548] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 100/100 [01:36<00:00,  1.04it/s, est. speed input: 573.55 toks/s, output: 106.52 toks/s]

Remaining parse failures: 0


,QuestionId,Question.Text,MC_Answer,StudentExplanation,Misconception_Code,split,label_is_correct,label_is_misunderstanding,is_correct,is_misunderstanding,which_misunderstanding
0,33472,\( \frac{1}{3}+\frac{2}{5}= \),\( \frac{3}{15} \),3 and 5 go into 15 so the domminater is 15 and...,Denominator-only change,train,False,True,False,True,The student demonstrates a **procedural misund...
1,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),the dots are going up in 4,True_Correct,train,True,False,False,True,The student recognized a pattern in the number...
2,91695,Dots have been arranged in these patterns: [Im...,\( 26 \),because it goes up by five and then u add two ...,True_Neither,train,False,False,False,True,The student appears to have a conceptual misun...
3,31777,A box contains \( 120 \) counters. The counter...,\( 72 \),I think this because 1/5 of 120 is 24 and 3/5 ...,True_Correct,train,True,False,True,False,
4,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 3 \),i was just guessing because i haven't gone ove...,False_Neither,train,False,False,False,False,
5,91695,Dots have been arranged in these patterns: [Im...,\( 22 \),So we addd four in of them 18+4 in total.,Wrong term,train,False,True,True,False,
6,31772,What fraction of the shape is not shaded? Give...,\( \frac{3}{9} \),becausee there are 9 total and three are not s...,Incomplete,train,False,True,False,True,The student has a conceptual misunderstanding ...
7,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),you do what do you do to the bottom to the top...,Additive,train,False,True,False,True,The student appears to be applying a procedura...
8,31778,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,\( 6 \),6/10 is equivalent to 9/15 because 5x2=10 and ...,True_Correct,train,True,False,False,True,The student incorrectly believes that multiply...
9,32833,Calculate \( \frac{2}{3} \times 5 \),\( 3 \frac{1}{3} \),i think this is because i did 2 1/3 times 5/1 ...,True_Correct,train,True,False,False,True,The student incorrectly multiplied the mixed n...


In [5]:
results_df.to_csv("../results/results0.csv")

## Classification Report

In [34]:
####################################
# This is from the previous prompt #
####################################


# from sklearn.metrics import classification_report

# print("Classification for is_correct")
# print(classification_report(
#     results_df['label_is_correct'],
#     results_df['is_correct'],
# ))

# print("Classification for is_misunderstanding")
# print(classification_report(
#     results_df['label_is_misunderstanding'],
#     results_df['is_misunderstanding'],
# ))

Classification for is_correct
              precision    recall  f1-score   support

       False       0.74      0.86      0.79        56
        True       0.77      0.61      0.68        44

    accuracy                           0.75       100
   macro avg       0.75      0.74      0.74       100
weighted avg       0.75      0.75      0.75       100

Classification for is_misunderstanding
              precision    recall  f1-score   support

       False       0.90      0.53      0.67        72
        True       0.41      0.86      0.56        28

    accuracy                           0.62       100
   macro avg       0.66      0.69      0.61       100
weighted avg       0.77      0.62      0.64       100



In [6]:
from sklearn.metrics import classification_report

print("Classification for is_correct")
print(classification_report(
    results_df['label_is_correct'],
    results_df['is_correct'],
))

print("Classification for is_misunderstanding")
print(classification_report(
    results_df['label_is_misunderstanding'],
    results_df['is_misunderstanding'],
))

Classification for is_correct
              precision    recall  f1-score   support

       False       0.73      0.86      0.79        56
        True       0.76      0.59      0.67        44

    accuracy                           0.74       100
   macro avg       0.75      0.72      0.73       100
weighted avg       0.74      0.74      0.73       100

Classification for is_misunderstanding
              precision    recall  f1-score   support

       False       0.90      0.50      0.64        72
        True       0.40      0.86      0.55        28

    accuracy                           0.60       100
   macro avg       0.65      0.68      0.59       100
weighted avg       0.76      0.60      0.62       100

